# Garden City revamp

This is the original Garden City arrangement drawn explicitly on a 5 m grid. Each footprint and door below can be edited directly, without loading a GeoPackage or running a conversion algorithm. The main ring streets are two cells (10 m) wide; intersections and openings can be wider.

The original building partitions and entrances are retained. Narrowing the principal streets reduces the overall city from 330 m × 330 m to 290 m × 290 m; building sizes at those street-aligned grid lines can also change. Building IDs are based on the new grid coordinates and therefore differ from the original city.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from shapely.geometry import box
from nomad.city_gen import City

CELL_METERS = 5
CITY_DIMENSIONS = (58, 58)  # 290 m × 290 m


In [ ]:
# Each row is (door cell, footprint bounds). Coordinates are in 5 m cells.
# Bounds are (left, bottom, right, top); right/top are exclusive.
# Change these coordinates directly to adjust an individual building.
BUILDINGS = {
    "park": [
        ((35, 30), (23, 23, 35, 35)),
        ((9, 11), (10, 10, 13, 16)),  # Bottom-left garden
        ((43, 48), (42, 45, 48, 48)),  # Top-right garden
        ((3, 49), (2, 50, 8, 56)),  # Top-left retail garden: 30 m square
        ((51, 1), (50, 2, 56, 8)),  # Bottom-right retail garden: 30 m square
    ],
    "home": [
        ((21, 21), (18, 18, 21, 23)),
        ((24, 21), (21, 18, 26, 21)),
        ((27, 21), (26, 18, 29, 21)),
        ((30, 21), (29, 18, 32, 21)),
        ((35, 17), (35, 18, 37, 21)),
        ((38, 17), (37, 18, 40, 21)),
        ((36, 21), (37, 21, 40, 23)),
        ((36, 24), (37, 23, 40, 26)),
        ((36, 30), (37, 29, 40, 32)),
        ((36, 33), (37, 32, 40, 35)),
        ((40, 35), (37, 35, 40, 37)),
        ((35, 36), (35, 37, 40, 40)),
        ((33, 36), (32, 37, 35, 40)),
        ((30, 36), (29, 37, 32, 40)),
        ((24, 36), (23, 37, 26, 40)),
        ((21, 36), (21, 37, 23, 40)),
        ((19, 40), (18, 37, 21, 40)),
        ((17, 35), (18, 35, 21, 37)),
        ((21, 33), (18, 32, 21, 35)),
        ((21, 27), (18, 23, 21, 29)),
    ],
    "workplace": [
        ((14, 9), (13, 10, 16, 16)),
        ((16, 16), (16, 10, 21, 16)),
        ((21, 16), (21, 10, 26, 16)),
        ((33, 16), (29, 13, 37, 16)),
        ((33, 9), (29, 10, 37, 13)),
        ((40, 9), (37, 10, 45, 16)),
        ((48, 11), (45, 10, 48, 16)),
        ((48, 16), (42, 16, 48, 21)),
        ((41, 24), (42, 21, 45, 26)),
        ((48, 21), (45, 21, 48, 26)),
        ((48, 27), (42, 26, 48, 32)),
        ((48, 35), (42, 35, 48, 40)),
        ((48, 40), (42, 40, 48, 42)),
        ((40, 41), (40, 42, 48, 45)),
        ((38, 41), (37, 42, 40, 45)),
        ((40, 48), (37, 45, 42, 48)),
        ((35, 48), (32, 42, 37, 48)),
        ((30, 48), (26, 45, 32, 48)),
        ((30, 41), (26, 42, 32, 45)),
        ((21, 48), (18, 42, 23, 48)),
        ((16, 48), (13, 45, 18, 48)),
        ((16, 41), (13, 42, 18, 45)),
        ((9, 43), (10, 42, 13, 48)),
        ((9, 35), (10, 35, 16, 42)),
        ((16, 33), (10, 32, 16, 35)),
        ((9, 27), (10, 23, 13, 29)),
        ((16, 24), (13, 23, 16, 29)),
        ((16, 21), (10, 21, 16, 23)),
        ((9, 16), (10, 16, 16, 21)),
    ],
    "retail": [
        ((49, 8), (50, 8, 56, 10)),  # Remainder beside bottom-right garden
        ((1, 3), (2, 2, 8, 8)),
        ((8, 1), (8, 2, 13, 8)),
        ((14, 1), (13, 2, 16, 5)),
        ((14, 8), (13, 5, 16, 8)),
        ((16, 1), (16, 2, 21, 5)),
        ((16, 8), (16, 5, 21, 8)),
        ((24, 8), (23, 2, 26, 8)),
        ((33, 8), (26, 2, 35, 8)),
        ((38, 8), (35, 2, 40, 8)),
        ((40, 8), (40, 5, 42, 8)),
        ((43, 8), (42, 5, 45, 8)),
        ((40, 1), (40, 2, 42, 5)),
        ((43, 1), (42, 2, 45, 5)),
        ((46, 8), (45, 5, 50, 8)),
        ((48, 1), (45, 2, 50, 5)),
        ((49, 14), (50, 10, 56, 16)),
        ((49, 19), (50, 16, 53, 21)),
        ((56, 19), (53, 16, 56, 21)),
        ((49, 27), (50, 23, 56, 29)),
        ((49, 30), (50, 29, 56, 35)),
        ((49, 35), (50, 35, 53, 40)),
        ((56, 35), (53, 35, 56, 40)),
        ((56, 43), (50, 40, 56, 45)),
        ((56, 48), (50, 45, 56, 50)),
        ((56, 51), (50, 50, 56, 53)),
        ((54, 56), (50, 53, 56, 56)),
        ((46, 49), (45, 50, 48, 56)),
        ((43, 49), (42, 50, 45, 56)),
        ((38, 49), (35, 50, 42, 53)),
        ((40, 56), (37, 53, 42, 56)),
        ((35, 56), (32, 53, 37, 56)),
        ((33, 49), (32, 50, 35, 53)),
        ((30, 49), (26, 50, 32, 56)),
        ((24, 49), (21, 50, 26, 53)),
        ((24, 56), (21, 53, 26, 56)),
        ((16, 56), (13, 50, 18, 56)),
        ((11, 56), (8, 53, 13, 56)),
        ((11, 49), (8, 50, 13, 53)),
        ((8, 46), (2, 42, 8, 48)),
        ((8, 40), (2, 40, 8, 42)),
        ((8, 38), (2, 37, 8, 40)),
        ((8, 33), (2, 32, 8, 37)),
        ((8, 30), (2, 29, 8, 32)),
        ((8, 27), (2, 26, 8, 29)),
        ((8, 21), (2, 21, 8, 26)),
        ((8, 19), (2, 18, 8, 21)),
        ((1, 14), (2, 10, 5, 18)),
        ((8, 16), (5, 16, 8, 18)),
        ((8, 14), (5, 13, 8, 16)),
        ((8, 11), (5, 10, 8, 13)),
    ],
}

fine_city = City(dimensions=CITY_DIMENSIONS, block_side_length=CELL_METERS,
                 name="Garden City Revamp")
for building_type, buildings in BUILDINGS.items():
    for door, bounds in buildings:
        fine_city.add_building(building_type, door=door, geom=box(*bounds))

# Keep the name used by the original Garden City notebook as well.
city = fine_city


In [ ]:
print(f"Buildings: {len(fine_city.buildings_gdf)}")
fig, ax = plt.subplots(figsize=(10, 10))
fine_city.plot_city(ax, doors=True, address=False, legend=True)
ax.set_title("Garden City revamp — 5 m cells")
ax.set_aspect("equal")
plt.show()


In [ ]:
# Verify that all entrances lead to the connected street network.
G = fine_city.get_street_graph()
assert nx.is_connected(G), "Street network is disconnected"
assert all((int(row.door_cell_x), int(row.door_cell_y)) in G
           for row in fine_city.buildings_gdf.itertuples(index=False)), "A door is not on a street"
print(f"Connected street network: {G.number_of_nodes()} cells; all doors accessible")

# Optional: save under a new name after inspecting the city.
# fine_city.save_geopackage('../garden-city-revamp.gpkg', persist_gravity_data=False)
